In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -U transformers
!pip install peft

In [ ]:
import torch
import numpy as np
import random
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    set_seed
)
from peft import get_peft_model, LoraConfig, TaskType
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
SEED = 42
BATCH_SIZE = 8
MAX_LENGTH = 512
NUM_EPOCHS = 50
LEARNING_RATE = 2e-4
PATIENCE = 5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

In [ ]:
TRAIN_FILE = "/content/drive/My Drive/italian_poems/italian_poems_val_autorship.json"
TEST_FILE = "/content/drive/My Drive/italian_poems/italian_poems_test_autorship.json"

MODELS = [
    "dbmdz/bert-base-italian-xxl-cased",
    "linhd-postdata/alberti-bert-base-multilingual-cased",
    "mattiaferrarini/saba"
]

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def fix_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)

In [ ]:
dataset = load_dataset("json", data_files={"train": TRAIN_FILE, "test": TEST_FILE})
test_split = dataset["test"].train_test_split(test_size=0.8, seed=SEED)

dataset_splits = {
    "train": dataset["train"],
    "validation": test_split["train"],
    "test": test_split["test"]
}

In [ ]:
authors = sorted(list(set(dataset_splits["train"]["author"])))
num_labels = len(authors)
label2id = {author: i for i, author in enumerate(authors)}
id2label = {i: author for i, author in enumerate(authors)}

print(f"Authors: {len(authors)}")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="macro")
    return {"accuracy": acc, "f1_macro": f1}

In [ ]:
for model_id in MODELS:
    print(f"\n{'='*20}")
    print(f"Processing: {model_id}")
    print(f"{'='*20}")

    fix_seed(SEED)
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    def preprocess_function(examples):
        tokenized = tokenizer(
            examples["text"],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False
        )
        tokenized["label"] = [label2id[author] for author in examples["author"]]
        return tokenized

    encoded_splits = {
        split: data.map(preprocess_function, batched=True)
        for split, data in dataset_splits.items()
    }

    # Initialize Model
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id
    ).to(device)

    # Apply LoRA
    peft_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        inference_mode=False,
        r=16,
        lora_alpha=32,
        lora_dropout=0.1,
        target_modules=["query", "value"]
    )
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

    # Training Args
    args = TrainingArguments(
        output_dir=f"./results_lora_cm_{model_id.replace('/', '_')}",
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type="cosine",
        save_total_limit=1,
        seed=SEED,
        fp16=torch.cuda.is_available(),
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=encoded_splits["train"],
        eval_dataset=encoded_splits["validation"],
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
    )

    # Train
    trainer.train()

    # Evaluation
    print(f"Generating Confusion Matrix for {model_id}...")

    test_output = trainer.predict(encoded_splits["test"])
    y_true = test_output.label_ids
    y_pred = np.argmax(test_output.predictions, axis=1)

    final_acc = accuracy_score(y_true, y_pred)
    final_f1 = f1_score(y_true, y_pred, average="macro")
    print(f"Final Test Accuracy: {final_acc:.4f} | F1: {final_f1:.4f}")

    cm = confusion_matrix(y_true, y_pred, labels=range(num_labels))
    fig, ax = plt.subplots(figsize=(10, 10))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=[id2label[i] for i in range(num_labels)]
    )

    disp.plot(ax=ax, cmap="Blues", xticks_rotation="vertical", values_format="d")
    plt.title(f"Confusion Matrix: {model_id}\n(Acc: {final_acc:.2f}, F1: {final_f1:.2f})")
    plt.tight_layout()
    plt.show() 
    
    # Cleanup
    del model
    del trainer
    torch.cuda.empty_cache()